# Laboration 2

## Excercise 1

In [ ]:
from typing import List, Optional
import gymnasium as gym
import pygame
import numpy as np

from collections import namedtuple, deque
from pygame.colordict import THECOLORS as CM

Text = namedtuple("Text", ("message", "position", "color"))


class LunarGame:
    WIDTH: int = 600
    HEIGHT: int = 400
    FPS: int = 30
    CONTROL_MSG: Text = Text("Controls: LEFT, UP, RIGHT", (20, 20), CM["cadetblue"])
    START_MSG: Text = Text("Press 'R' to start.", (20, 20), CM["yellow"])
    LINDQVIST: float = 37.84

    def __init__(self):
        pygame.init()
        pygame.display.set_caption("Lunar Lander Control")
        self.screen = pygame.display.set_mode((LunarGame.WIDTH, LunarGame.HEIGHT))
        self.clock = pygame.time.Clock()
        self.ten_latest = deque(maxlen=10)
        self.env = gym.make("LunarLander-v3", render_mode="rgb_array")
        self.env.reset()

    def run(self):
        q = self._menu([LunarGame.START_MSG])
        while not q:
            if (g := self._episode_loop()) is None:
                break
            q = self._menu(
                [
                    LunarGame.START_MSG,
                ]
                + self._gen_menu_stats(g)
            )

    def _display_text(self, text: Text):
        font = pygame.font.SysFont("Arial", 18)
        text_surface = font.render(text.message, True, text.color)
        self.screen.blit(text_surface, text.position)

    def _render_frame(self):
        img = self.env.render()
        img = np.rot90(img)
        img_surface = pygame.surfarray.make_surface(img)
        self.screen.blit(img_surface, (0, 0))
        self._display_text(LunarGame.CONTROL_MSG)
        pygame.display.flip()
        self.clock.tick(LunarGame.FPS)

    def _read_action() -> int:
        keys = pygame.key.get_pressed()
        if keys[pygame.K_LEFT]:
            action = 1  # LEFT ENGINE
        elif keys[pygame.K_UP]:
            action = 2  # MAIN ENGINE
        elif keys[pygame.K_RIGHT]:
            action = 3  # RIGHT ENGINE
        else:
            action = 0
        return action

    def _menu(self, textlst: List[Text]) -> bool:
        self.screen.fill(CM["black"])
        r_pressed = False
        for text in textlst:
            self._display_text(text)
        pygame.display.flip()
        while not r_pressed:
            if self._check_quit():
                return True
            self.clock.tick(LunarGame.FPS)
            r_pressed = pygame.key.get_pressed()[pygame.K_r]
        return False

    def _check_quit(self) -> bool:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                self.env.close()
                return True
        return False

    def _episode_loop(self) -> Optional[float]:
        episode_return = 0
        while True:
            if self._check_quit():
                return None
            action = LunarGame._read_action()
            _, reward, terminated, _, _ = self.env.step(action)
            episode_return += reward
            self._render_frame()
            if terminated:
                self.env.reset()
                break
        return episode_return

    def _gen_menu_stats(self, g: float) -> List[Text]:
        self.ten_latest.append(g)
        num_samples = len(self.ten_latest)
        ma5 = np.mean(self.ten_latest)
        ma5str = "MA10: " + (
            f"{ma5:.2f}"
            if num_samples == 10
            else f"N/A (only {num_samples} sample"
            + ("s" if num_samples > 1 else "")
            + ")"
        )
        ma5color = CM["red"] if ma5 < LunarGame.LINDQVIST else CM["green"]
        hist1 = f"Latest: {np.round(list(self.ten_latest)[:5],2)}"
        hist2 = (
            " " * 17 + f"{np.round(list(self.ten_latest)[5:],2)}"
            if num_samples > 5
            else ""
        )
        return [
            Text(s, (20, 60 + i * 40), c)
            for i, (s, c) in enumerate(
                [
                    (f"Last return: {g:.2f}", CM["white"]),
                    (ma5str, ma5color),
                    (hist1, CM["white"]),
                    (hist2, CM["white"]),
                ]
            )
        ]

# game = LunarGame()
# game.run()
# print(
#     f"Game exited.\nLatest: {np.round(list(game.ten_latest), 2)}\nAvg: {np.mean(game.ten_latest):.2f}"
# )

Game exited.
Latest: [275.93 238.46  20.77 -25.33 246.61  10.32 218.04 201.71 -83.81 242.27]
Avg: 134.50


## Excercise 2

In [ ]:
from torch import nn

class LanderAgent(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        if len(hidden) < 2:
            raise ValueError("hidden must contain input and output dimensions")
        self.hidden = tuple(hidden)

        # Input for LunarLander is 8-dimensional, output is 4-dimensional (4 actions)
        self.layers = nn.ModuleList(
            [nn.Linear(hidden[i - 1], hidden[i]) for i in range(1, len(hidden))]
        )


## Excercise 3